In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, accuracy_score, roc_curve, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import xgboost as xgb

# Bayesian
import pymc as pm
import arviz as az

# Display options
pd.set_option('display.max_columns', 200)
np.random.seed(42)
import arviz as az

## Load Data and Yield Pandas DataFrame

In [ ]:
df=pd.read_excel("IPL_Bowler_Detailed_Data.xls")
df.head()

## Data Description
##### The dataset contains ball-by-ball cricket information, where each row represents a single delivery.

- Match_ID and Match_Date identify the match in which the ball was bowled.

- Pitch_Type describes the surface conditions that influence scoring and wicket probability.

- Phase indicates the innings stage (Powerplay, Middle Overs, or Death).

- Over and Ball together pinpoint the exact delivery sequence in the innings.

- Bowler specifies who delivered the ball, enabling skill-based analysis.

- Batter_Avg gives the batter’s average, representing their consistency in scoring runs.

- Batter_SR provides the batter’s strike rate, indicating their scoring speed.

- Runs_Conceded records how many runs were scored from that specific delivery.

- Is_Wicket shows whether the ball resulted in a wicket.


## Data cleaning

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

In [ ]:
df.describe(include='all').T

In [ ]:
df.shape

In [ ]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
df.info()

## standardization

In [ ]:
# Ensure column names expected by this notebook:
expected_cols = ['Match_ID','Match_Date','Pitch_Type','Phase','Over','Ball','Bowler', 'Batter_Avg','Batter_SR','Runs_Conceded','Is_Wicket']
# Rename if necessary:
# df.rename(columns={'batter_avg':'Batter_Avg', ...}, inplace=True)
for c in expected_cols:
    if c not in df.columns:
        print("Warning: missing column", c)

In [ ]:
# Convert types
df['Match_Date'] = pd.to_datetime(df['Match_Date'], errors='coerce')
# Ensure numeric types
df['Over'] = pd.to_numeric(df['Over'], errors='coerce').astype(int)
df['Ball'] = pd.to_numeric(df['Ball'], errors='coerce').astype(int)
df['Runs_Conceded'] = pd.to_numeric(df['Runs_Conceded'], errors='coerce').fillna(0).astype(int)
df['Is_Wicket'] = pd.to_numeric(df['Is_Wicket'], errors='coerce').fillna(0).astype(int)

In [ ]:
# Filter Death overs (only consider overs 16-20)
df = df[df['Phase'].str.lower().str.contains('death', na=False) | df['Over'].isin([16,17,18,19,20])]
df = df.sort_values(['Match_ID','Over','Ball']).reset_index(drop=True)
print("After filtering Death overs: ", len(df))

In [ ]:
# Feature Engineering: Dot ball and Pressure (correct logic)
# Dot ball indicator
df['Dot_Ball'] = (df['Runs_Conceded'] == 0).astype(int)

In [ ]:
# Create Pressure: 1 if previous delivery (same match, same over) was dot and bowler same (or if you want irrespective of bowler - see below).
# Important: previous ball must be same Match_ID and same Over. Also ensure no cross-over or over-boundary.
df['prev_Match_ID'] = df['Match_ID'].shift(1)
df['prev_Over'] = df['Over'].shift(1)
df['prev_Ball'] = df['Ball'].shift(1)
df['prev_Bowler'] = df['Bowler'].shift(1)
df['prev_Dot'] = df['Dot_Ball'].shift(1)

df['Pressure'] = (
    (df['prev_Match_ID'] == df['Match_ID']) &
    (df['prev_Over'] == df['Over']) &
    (df['prev_Dot'] == 1)
).astype(int)


In [ ]:
#If you give importance to pressure only when same bowler bowled previous ball:
df['Pressure_same_bowler'] = df['Pressure'] & (df['prev_Bowler'] == df['Bowler'])

In [ ]:
# Clean helper cols
df.drop(['prev_Match_ID','prev_Over','prev_Ball','prev_Bowler','prev_Dot'], axis=1, inplace=True)

df[['Match_ID','Over','Ball','Bowler','Runs_Conceded','Dot_Ball','Pressure','Is_Wicket']].head(20)

In [ ]:
# Encode Pitch_Type roughly
df['Pitch_Type'] = df['Pitch_Type'].fillna('Neutral')
# Simplify pitch labels if needed
def pitch_map(x):
    x = str(x).lower()
    if 'bat' in x:
        return 'Batting'
    if 'bowl' in x or 'green' in x or 'dust' in x:
        return 'Bowling'
    return 'Neutral'
df['Pitch_Simple'] = df['Pitch_Type'].apply(pitch_map)

In [ ]:
# Standardize Batter_Avg - fill missing with median
df['Batter_Avg'] = pd.to_numeric(df['Batter_Avg'], errors='coerce')
df['Batter_Avg'] = df['Batter_Avg'].fillna(df['Batter_Avg'].median())

In [ ]:
# 1 if Bowler B, 0 if A (adjust if names differ)

In [ ]:
# Bowler indicator (two bowlers A and B expected)
df['Bowler'] = df['Bowler'].astype(str)
df['Bowler_code'] = (df['Bowler'] == 'B').astype(int)

In [ ]:
## # Make sure last ball of an over does not create Pressure for next over
bad = df[(df['Pressure']==1) & ( (df['Over'] != df['Over'].shift(1)) | (df['Match_ID'] != df['Match_ID'].shift(1)) )]
print("Bad pressure rows (should be 0):", len(bad))

In [ ]:
# Basic aggregated stats
agg = df.groupby('Bowler')[['Dot_Ball','Pressure','Is_Wicket']].mean().reset_index()
agg.columns = ['Bowler','Avg_Dot_Ball','Avg_Pressure','Wicket_Rate']
agg

## EDA: plots

In [ ]:
sns.set(style="whitegrid")

In [ ]:
# Bar: pressure vs non-pressure wicket %
pt = df.groupby('Pressure')['Is_Wicket'].mean().reset_index()
plt.figure(figsize=(5,4))
sns.barplot(data=pt, x='Pressure', y='Is_Wicket')
plt.title('Wicket probability: Pressure vs Non-Pressure')
plt.ylabel('P(Is_Wicket)')
plt.xlabel('Pressure on this ball (prev same-over dot)')
plt.show()

In [ ]:
#  Per-bowler pressure effect
plt.figure(figsize=(7,4))
sns.barplot(data=df, x='Bowler', y='Is_Wicket', hue='Pressure', ci=None)
plt.title('Wicket rate by Bowler and Pressure')
plt.show()

In [ ]:
# Distribution of Batter_Avg by Pressure (violin)
plt.figure(figsize=(6,4))
sns.violinplot(data=df, x='Pressure', y='Batter_Avg')
plt.title('Batter_Avg distribution: Pressure vs Non-Pressure')
plt.show()

In [ ]:
# Heatmap: dotball & wickets by over ball position within death overs
pivot = df.pivot_table(index='Over', columns='Ball', values='Is_Wicket', aggfunc='mean')
plt.figure(figsize=(10,4))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap='YlGnBu')
plt.title('Wicket probability: Over x Ball')
plt.show()

#  Prepare dataset for ML

In [ ]:
features = ['Pressure','Pitch_Simple','Batter_Avg','Bowler_code']
X = df[features].copy()
y = df['Is_Wicket'].copy()

In [ ]:
# One-hot encode Pitch_Simple
X = pd.get_dummies(X, columns=['Pitch_Simple'], drop_first=True)

In [ ]:
# Train/test split stratified by y
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train.shape, X_test.shape

# Model evaluation function

In [ ]:
def evaluate_model(clf, X_train, y_train, X_test, y_test, name='Model'):
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X_test)[:,1]
    elif hasattr(clf, "decision_function"):
        proba = clf.decision_function(X_test)
        # scale to 0-1 roughly using minmax
        proba = (proba - proba.min()) / (proba.max() - proba.min() + 1e-12)
    else:
        proba = preds
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, proba) if len(np.unique(y_test))>1 else np.nan
    print(f"{name} - Acc: {acc:.3f}  ROC-AUC: {auc:.3f}")
    print(classification_report(y_test, preds, zero_division=0))
    return {'name': name, 'acc':acc, 'auc':auc}

In [ ]:
results = []

In [ ]:
# Logistic
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
results.append(evaluate_model(lr, X_train, y_train, X_test, y_test, 'LogisticRegression'))

In [ ]:
# RandomForest
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
results.append(evaluate_model(rf, X_train, y_train, X_test, y_test, 'RandomForest'))

In [ ]:
# XGBoost
xg = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
results.append(evaluate_model(xg, X_train, y_train, X_test, y_test, 'XGBoost'))

In [ ]:
# SVM (with probability)
svm = SVC(probability=True, class_weight='balanced', random_state=42)
results.append(evaluate_model(svm, X_train, y_train, X_test, y_test, 'SVM'))


In [ ]:
pd.DataFrame(results)

In [ ]:
# Feature importance (RF) and simple partial dependence of Pressure
fi = pd.Series(rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("Feature importances (RandomForest):")
print(fi)

In [ ]:

# Pressure effect by decile
grp = df.groupby('Pressure')['Is_Wicket'].agg(['mean','count']).reset_index()
grp

## Bayesian simple logistic GLM with interaction Bowler x Pressure

In [ ]:
# We'll fit a model on the whole dataset including Bowler_code and Pressure and their interaction.
data = df.copy()
# Standardize Batter_Avg for better sampling
data['Batter_Avg_s'] = (data['Batter_Avg'] - data['Batter_Avg'].mean()) / data['Batter_Avg'].std()

In [ ]:
# design matrix
data['Pressure_x_Bowler'] = data['Pressure'] * data['Bowler_code']

# Bayesian simple logistic GLM with interaction Bowler x Pressure

In [ ]:
 
# We'll fit a model on the whole dataset including Bowler_code and Pressure and their interaction.
data = df.copy()
# Standardize Batter_Avg for better sampling
data['Batter_Avg_s'] = (data['Batter_Avg'] - data['Batter_Avg'].mean()) / data['Batter_Avg'].std()

# design matrix
data['Pressure_x_Bowler'] = data['Pressure'] * data['Bowler_code']

with pm.Model(coords=coords) as model:
    # Priors
    alpha = pm.Normal('alpha', mu=0, sigma=2)
    b_pressure = pm.Normal('b_pressure', mu=0, sigma=2)
    b_bowler = pm.Normal('b_bowler', mu=0, sigma=2)
    b_inter = pm.Normal('b_inter', mu=0, sigma=2)
    b_bavg = pm.Normal('b_bavg', mu=0, sigma=2)

    # linear predictor
    logit_p = (alpha +
               b_pressure * data['Pressure'].values +
               b_bowler * data['Bowler_code'].values +
               b_inter * data['Pressure_x_Bowler'].values +
               b_bavg * data['Batter_Avg_s'].values)

    p = pm.Deterministic('p', pm.math.sigmoid(logit_p))

    # likelihood
    y_obs = pm.Bernoulli('y_obs', p=p, observed=data['Is_Wicket'].values)

    # sample
    trace_glm = pm.sample(draws=2000, tune=1000, target_accept=0.9, random_seed=42, chains=2)


In [ ]:
# Save Bayesian model results
trace_glm.to_netcdf('bayesian_glm_trace.nc')  # ArviZ format for PyMC
pd.DataFrame({
    'effect_A_mean': [effect_A.mean()], 
    'effect_A_hdi': [hdi_A.tolist()],
    'effect_B_mean': [effect_B.mean()], 
    'effect_B_hdi': [hdi_B.tolist()]
}).to_pickle('killer_instinct_results.pkl')

# Posterior summaries & HDI for Pressure terms

In [ ]:
az.summary(trace_glm, var_names=['b_pressure','b_bowler','b_inter','b_bavg','alpha'], hdi_prob=0.94)

In [ ]:
# Plot posterior for pressure and interaction
az.plot_posterior(trace_glm, var_names=['b_pressure','b_inter'], hdi_prob=0.94)
plt.show()

# Interpret interaction: pressure effect for Bowler A and Bowler B

- effect for Bowler A = b_pressure
- effect for Bowler B = b_pressure + b_inter

In [ ]:
b_pressure_samples = trace_glm.posterior['b_pressure'].stack(draws=("chain","draw")).values.flatten()
b_inter_samples = trace_glm.posterior['b_inter'].stack(draws=("chain","draw")).values.flatten()


In [ ]:
effect_A = b_pressure_samples
effect_B = b_pressure_samples + b_inter_samples

In [ ]:
def summarize_samples(samples, name):
    mean = np.mean(samples)
    hdi = az.hdi(samples, hdi_prob=0.94)
    print(f"{name} mean: {mean:.4f}, 94% HDI: [{hdi[0]:.4f}, {hdi[1]:.4f}]")

In [ ]:
summarize_samples(effect_A, "Pressure effect - Bowler A")
summarize_samples(effect_B, "Pressure effect - Bowler B")


In [ ]:
# Plot both posterior distributions
plt.figure(figsize=(7,4))
sns.kdeplot(effect_A, label='Bowler A', bw_method=0.3)
sns.kdeplot(effect_B, label='Bowler B', bw_method=0.3)
plt.legend()
plt.title('Posterior: Pressure effect (log-odds) for A vs B')
plt.xlabel('Coefficient (log-odds)')
plt.show()

In [ ]:
#Posterior predictive checks (PPC)
with glm_model:
    ppc = pm.sample_posterior_predictive(trace_glm, var_names=['y_obs'], random_seed=42)

# Extract from posterior_predictive group
y_sim = ppc.posterior_predictive['y_obs']

# Compute mean
ppc_mean = y_sim.mean(axis=0)

print("Observed wicket rate:", data['Is_Wicket'].mean())
print("Mean predicted wicket rate (ppc):", ppc_mean.mean())


#  Hierarchical model: Bowler-level partial pooling for pressure effect

In [ ]:
import pymc as pm
import numpy as np
from sklearn.preprocessing import StandardScaler

unique_bowlers = data['Bowler'].unique()
coords = {"bowler": unique_bowlers}
bowler_index_map = {b: i for i, b in enumerate(unique_bowlers)}
data['bowler_idx'] = data['Bowler'].map(bowler_index_map)

# **CRITICAL: Standardize continuous predictors**
scaler_pressure = StandardScaler()
scaler_bavg = StandardScaler()

pressure_scaled = scaler_pressure.fit_transform(data[['Pressure']]).flatten().astype(np.float32)
bavg_scaled = scaler_bavg.fit_transform(data[['Batter_Avg_s']]).flatten().astype(np.float32)

print(f"Pressure scaled: mean={pressure_scaled.mean():.3f}, std={pressure_scaled.std():.3f}")
print(f"Bavg scaled: mean={bavg_scaled.mean():.3f}, std={bavg_scaled.std():.3f}")

In [ ]:
with pm.Model(coords=coords) as model:
    # **SCALED DATA CONTAINERS**
    bowler_idx = pm.Data("bowler_idx", data["bowler_idx"].values.astype(np.int32))
    pressure = pm.Data("pressure", pressure_scaled)  # Standardized
    bavg = pm.Data("bavg", bavg_scaled)             # Standardized
    
    # **TIGHTER HYPERPRIORS** (for scaled data)
    mu_a = pm.Normal("mu_a", 0, 0.5)
    sigma_a = pm.HalfNormal("sigma_a", 0.5)  # More stable than Exponential
    
    mu_bp = pm.Normal("mu_bp", 0, 0.5)
    sigma_bp = pm.HalfNormal("sigma_bp", 0.5)
    
    # BOWLER-LEVEL EFFECTS
    a = pm.Normal("a", mu=mu_a, sigma=sigma_a, dims="bowler")
    bp = pm.Normal("bp", mu=mu_bp, sigma=sigma_bp, dims="bowler")
    
    # GLOBAL EFFECTS (tighter prior for scaled data)
    b_bavg = pm.Normal("b_bavg", 0, 1)
    
    # **EXPECTED LINEAR MODEL** (sigmoid expects logit scale)
    logit_p = a[bowler_idx] + bp[bowler_idx] * pressure + b_bavg * bavg
    
    p = pm.Deterministic("p", pm.math.sigmoid(logit_p))
    y_obs = pm.Bernoulli("y_obs", p=p, observed=data["Is_Wicket"].values.astype(np.float32))
    
    # **CONVERGENCE-OPTIMIZED SAMPLING**
    idata_h = pm.sample(
        draws=2000, 
        tune=2000,  # More tuning
        target_accept=0.95,  # Higher acceptance
        init='adapt_diag',
        cores=1,  # Single core stability
        random_seed=42
    )

# Diagnostics
print(az.summary(idata_h, hdi_prob=0.95).round(3))
 

In [ ]:
# Bowler rankings (posterior means)
bowler_effects = idata_h.posterior['a'].mean(dim=['chain', 'draw']).values
print("\nTop Bowlers (wicket probability effect):")
top_bowlers = pd.DataFrame({
    'bowler': unique_bowlers,
    'effect': bowler_effects
}).sort_values('effect', ascending=False).head(10)
print(top_bowlers)


In [ ]:
# Summarize pitch-level pressure slopes
print("\n--- Pitch-level pressure slope summaries (94% HDI) ---")
for i, name in enumerate(pitch_names):
    samples = idata_pitch.posterior['bp'].sel({ 'bp_dim_0': i }).stack(draws=("chain","draw")).values.flatten()
    hdi = az.hdi(samples, hdi_prob=0.94)
    print(f"Pitch: {name:8s} -> mean {samples.mean():.4f}, 94% HDI [{hdi[0]:.4f}, {hdi[1]:.4f}]")


## Markov chain simulation of over sequences (dot / non-dot / wicket)

- Goal: model the sequence of deliveries within the death overs as a Markov chain so you can simulate sequences (and simulate how often dot→wicket occurs) and compare bowlers.

States:

### Use a compact state space per delivery:

- D = Dot ball (0 runs, non-wicket)

- R = Run (1+ runs, non-wicket)

- W = Wicket

- (You can extend to R1, R2, 4, 6 if needed; for pressure focus D/R/W is fine.)

- Count transitions between consecutive same-over deliveries (no cross-over). For each pair of consecutive deliveries (prev -> curr), increment the matrix cell.

In [ ]:
# df sorted by Match_ID, Over, Ball
states = []
def state_for_row(r):
    if r.Is_Wicket==1: return 'W'
    return 'D' if r.Runs_Conceded==0 else 'R'

In [ ]:
# create state column
df['state'] = df.apply(state_for_row, axis=1)

In [ ]:
# only same-over consecutive transitions
df['prev_match'] = df['Match_ID'].shift(1)
df['prev_over'] = df['Over'].shift(1)
df['prev_state'] = df['state'].shift(1)

In [ ]:
mask = (df['Match_ID'] == df['prev_match']) & (df['Over'] == df['prev_over'])
transitions = df[mask].groupby(['prev_state','state']).size().unstack(fill_value=0)

In [ ]:
# Transition probability matrix
trans_mat = transitions.div(transitions.sum(axis=1), axis=0)
print(trans_mat)

### Simulate sequences

In [ ]:
def simulate_over(start_state='R', trans_mat=trans_mat, n_balls=6):
    states = [start_state]
    for i in range(1, n_balls):
        cur = states[-1]
        probs = trans_mat.loc[cur].values
        states.append(np.random.choice(trans_mat.columns, p=probs))
    return states

### simulate many overs and compute frequency of Dot->Wicket (prev D then W)
 

In [ ]:
N=20000
count=0
for _ in range(N):
    seq = simulate_over(start_state='R')
    # check any D followed by W inside the over
    for i in range(len(seq)-1):
        if seq[i]=='D' and seq[i+1]=='W':
            count += 1
            break
count/N

## Transition matrix heatmap (annotated probabilities).

In [ ]:
# 1. Transition matrix heatmap
plt.figure(figsize=(8,6))
sns.heatmap(trans_mat, annot=True, cmap='Blues', fmt='.3f', cbar_kws={'label': 'Probability'})
plt.title('Cricket State Transition Matrix')
plt.ylabel('Current State')
plt.xlabel('Next State')
plt.tight_layout()
plt.show()

### Simulated distribution of number of wickets per over (histogram).

In [ ]:
# 2. Wickets per over histogram (simulate first)
wickets_per_over = []
for _ in range(N):
    seq = simulate_over('R')
    wickets = seq.count('W')
    wickets_per_over.append(wickets)
plt.figure(figsize=(8,5))
plt.hist(wickets_per_over, bins=range(8), edgecolor='black', alpha=0.7)
plt.title('Simulated Wickets per Over Distribution')
plt.xlabel('Wickets per Over')
plt.ylabel('Frequency')
plt.show(

#### Observed vs simulated frequency of D→W transitions (bar chart) to validate model.

In [ ]:
# 3. Observed vs Simulated D→W frequency
simulated_freq = count/N
plt.figure(figsize=(6,5))
states = ['Simulated', 'Observed']  # Add your observed freq
freqs = [simulated_freq, observed_dw_freq]  # Replace with actual observed
plt.bar(states, freqs, color=['skyblue', 'orange'], alpha=0.8)
plt.title('D→W Transition Frequency Validation')
plt.ylabel('Frequency')
plt.ylim(0, max(freqs)*1.1)
for i, v in enumerate(freqs):
    plt.text(i, v+0.001, f'{v:.3f}', ha='center')
plt.show()

### Interpretation

- If simulated D→W frequency matches observed, Markov model is a reasonable generative model.

- If mismatch, consider higher-order Markov (use previous two states) or include covariates (bowler/pitch) by making transition probabilities conditional.

# Model wicket probability conditional on dot-sequence length

### Goal: quantify how the probability of a wicket on the next ball depends on how many consecutive preceding dot balls occurred (pressure accumulation).

##### Feature engineering

- Create a feature prev_dot_run = number of consecutive previous deliveries in the same over that were dot balls immediately preceding this delivery. Example sequence (same over):

- ball1: R → prev_dot_run = 0

- ball2: D → prev_dot_run = 0 (because this is the current ball)

- ball3: ? → if ball2 was D and ball1 was not, prev_dot_run = 1

- ball4: if ball2 & ball3 were D, prev_dot_run = 2, etc.

In [ ]:
# assume df sorted by Match_ID, Over, Ball
df['prev_is_dot'] = (df['Runs_Conceded'].shift(1)==0) & (df['Match_ID'].shift(1)==df['Match_ID']) & (df['Over'].shift(1)==df['Over'])
# We'll compute consecutive dots using groupby on match+over and scanning
def compute_prev_dot_run(group):
    prev_dot_run = []
    consec = 0
    for idx, row in group.iterrows():
        # consec currently holds how many consecutive dots BEFORE this ball
        prev_dot_run.append(consec)
        # update consec for next ball (current becomes previous)
        if row.Runs_Conceded == 0 and row.Is_Wicket==0:
            consec += 1
        else:
            consec = 0
    return pd.Series(prev_dot_run, index=group.index)

df['prev_dot_run_len'] = df.groupby(['Match_ID','Over']).apply(compute_prev_dot_run).reset_index(level=[0,1], drop=True)

### Check whether to compute consecutive previous dots (same over)

In [ ]:
# assume df sorted by Match_ID, Over, Ball
df['prev_is_dot'] = (df['Runs_Conceded'].shift(1)==0) & (df['Match_ID'].shift(1)==df['Match_ID']) & (df['Over'].shift(1)==df['Over'])
# We'll compute consecutive dots using groupby on match+over and scanning
def compute_prev_dot_run(group):
    prev_dot_run = []
    consec = 0
    for idx, row in group.iterrows():
        # consec currently holds how many consecutive dots BEFORE this ball
        prev_dot_run.append(consec)
        # update consec for next ball (current becomes previous)
        if row.Runs_Conceded == 0 and row.Is_Wicket==0:
            consec += 1
        else:
            consec = 0
    return pd.Series(prev_dot_run, index=group.index)

df['prev_dot_run_len'] = df.groupby(['Match_ID','Over']).apply(compute_prev_dot_run).reset_index(level=[0,1], drop=True)

#### Barplot: Probability of wicket given prev_dot_run_len = k with 95% binomial CIs

In [ ]:
prob_wicket = df.groupby('prev_dot_run_len')['Is_Wicket'].agg(['mean', 'count'])
prob_wicket['stderr'] = np.sqrt(prob_wicket['mean'] * (1 - prob_wicket['mean']) / prob_wicket['count'])
prob_wicket['ci95_hi'] = prob_wicket['mean'] + 1.96 * prob_wicket['stderr']
prob_wicket['ci95_lo'] = prob_wicket['mean'] - 1.96 * prob_wicket['stderr']

plt.figure(figsize=(10,6))
sns.barplot(x=prob_wicket.index, y=prob_wicket['mean'], color='skyblue')
plt.errorbar(prob_wicket.index, prob_wicket['mean'], 
             yerr=1.96*prob_wicket['stderr'], fmt='none', c='black', capsize=4)
plt.xlabel('Length of consecutive previous dot balls')
plt.ylabel('Probability of Wicket')
plt.title('P(Is_Wicket | prev_dot_run_len)')
plt.show()

#### Line plot: Probability vs prev_dot_run_len with error bars

In [ ]:
plt.figure(figsize=(10,6))
plt.errorbar(prob_wicket.index, prob_wicket['mean'], 
             yerr=1.96*prob_wicket['stderr'], fmt='-o', color='darkred', capsize=5)
plt.xlabel('Previous consecutive dot balls length')
plt.ylabel('Probability of wicket')
plt.title('Probability of wicket vs previous dot balls sequence length')
plt.grid(True)
plt.show()

#### Histogram: counts of deliveries by prev_dot_run_len (shows sparsity at high k).

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['prev_dot_run_len'], bins=range(0, df['prev_dot_run_len'].max() + 2), color='purple')
plt.xlabel('Length of previous consecutive dot balls')
plt.ylabel('Count of deliveries')
plt.title('Distribution of previous dot ball sequence lengths')
plt.show()

#### Simple conditional probability calculation

In [ ]:
grp = df.groupby('prev_dot_run_len')['Is_Wicket'].agg(['sum','count'])
grp['p_hat'] = grp['sum']/grp['count']
# compute Wilson CI or use statsmodels proportion_confint

# Modeling approaches

# Nonparametric / Empirical: just report conditional frequencies and confidence intervals.

- Logistic regression (ML): include prev_dot_run_len as a categorical or numeric predictor (maybe nonlinear).

- numeric: fit logit(p) = α + β * prev_dot_run_len.

- categorical: separate coefficient per length (k=0,1,2,3+) to capture nonlinearity.

- Bayesian logistic: put priors on coefficients — good for small counts at high k.

- Poisson / Survival view: model time-to-wicket in the over given dot runs (competing risk). (Advanced)
 

In [ ]:
# create categorical bins: 0,1,2,3,4+ (adjust)
df['dot_bin'] = df['prev_dot_run_len'].clip(upper=4)

import pymc as pm
with pm.Model() as dotlen_model:
    alpha = pm.Normal('alpha', 0, 2)
    # one coefficient per bin (except baseline 0)
    betas = pm.Normal('beta', mu=0, sigma=2, shape=4)  # bins 1..4
    # build linear predictor
    dot_idx = df['dot_bin'].values.astype(int)
    # map: if dot_bin==0 -> lp = alpha; else lp = alpha + betas[dot_bin-1]
    lp = alpha + pm.math.switch(dot_idx>0, betas[dot_idx-1], 0)
    p = pm.math.sigmoid(lp)
    y = pm.Bernoulli('y', p=p, observed=df['Is_Wicket'].values)
    trace_dot = pm.sample(2000, tune=1000, target_accept=0.9)

# Empirical vs Model: P(Is_Wicket | prev_dot_run_len) with 95% CI + model overlay

In [ ]:
prob_emp = df.groupby('dot_bin')['Is_Wicket'].agg(['mean', 'count'])
prob_emp['ci_lo'] = prob_emp['mean'] - 1.96 * np.sqrt(prob_emp['mean']*(1-prob_emp['mean'])/prob_emp['count'])
prob_emp['ci_hi'] = prob_emp['mean'] + 1.96 * np.sqrt(prob_emp['mean']*(1-prob_emp['mean'])/prob_emp['count'])

plt.figure(figsize=(10,6))
plt.errorbar(prob_emp.index, prob_emp['mean'], 
             yerr=[prob_emp['mean']-prob_emp['ci_lo'], prob_emp['ci_hi']-prob_emp['mean']], 
             fmt='o', capsize=5, label='Empirical (95% CI)', color='black')
# Model predictions (mean posterior)
post_means = np.mean(np.exp(trace_dot.posterior['alpha'] + np.maximum(trace_dot.posterior['beta'], 0)) / 
                    (1 + np.exp(trace_dot.posterior['alpha'] + np.maximum(trace_dot.posterior['beta'], 0))), axis=(0,1))
plt.plot(range(5), post_means, 'r-', linewidth=3, label='Model fit')
plt.xlabel('Previous dot balls (0,1,2,3,4+)')
plt.ylabel('P(Wicket)')
plt.title('Empirical vs Model: Wicket Probability by Dot Sequence Length')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Posterior distributions for beta coefficients (HDIs)

In [ ]:
az.plot_posterior(trace_dot, var_names=['beta'], hdi_prob=0.95, ref_val=0)
plt.suptitle('Posterior Distributions: β coefficients for dot bins 1-4')
plt.tight_layout()
plt.show()

#  Predicted probability vs k with posterior predictive intervals

In [ ]:
pred_grid = np.arange(5)
pred_probs = []
for k in pred_grid:
    lp_post = trace_dot.posterior['alpha'] + pm.math.switch(k>0, trace_dot.posterior['beta'][:,k-1], 0)
    p_post = 1 / (1 + np.exp(-lp_post))
    pred_probs.append(p_post.values.flatten())
pred_probs = np.array(pred_probs)

plt.figure(figsize=(10,6))
plt.plot(pred_grid, np.mean(pred_probs, axis=1), 'b-', linewidth=3, label='Posterior mean')
plt.fill_between(pred_grid, np.percentile(pred_probs, 2.5, axis=1), 
                np.percentile(pred_probs, 97.5, axis=1), alpha=0.3, color='blue', label='95% PPI')
plt.scatter(prob_emp.index, prob_emp['mean'], color='red', s=100, zorder=5, label='Data')
plt.xlabel('Previous dot balls (0,1,2,3,4+)')
plt.ylabel('Predicted P(Wicket)')
plt.title('Model Predictions with 95% Posterior Predictive Intervals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Interpretation

- Look for a monotonic increase in wicket probability with longer dot sequences — that is direct evidence of "pressure accumulation".

- If coefficients for k≥1 are positive and HDI excludes zero, that's evidence pressure length matters.

# Hierarchical Bayesian model for Pitch Type

Goal: pressure effect might differ by pitch class (Batting / Neutral / Bowling). Use a hierarchical model that estimates global effects and pitch-specific deviations (partial pooling).

Model idea (logistic, with random intercepts and random slopes)
logit
  

In [ ]:
# factorize pitch types
pitch_codes, pitch_names = pd.factorize(df['Pitch_Simple'])
df['pitch_idx'] = pitch_codes


In [ ]:
with pm.Model() as pitch_hier:
    # hyperpriors
    mu_a = pm.Normal('mu_a', 0, 2)
    sigma_a = pm.HalfNormal('sigma_a', 1)
    mu_bp = pm.Normal('mu_bp', 0, 2)
    sigma_bp = pm.HalfNormal('sigma_bp', 1)

In [ ]:
# pitch-level parameters
a = pm.Normal('a', mu=mu_a, sigma=sigma_a, shape=len(pitch_names))
bp = pm.Normal('bp', mu=mu_bp, sigma=sigma_bp, shape=len(pitch_names))

In [ ]:
# global coeffs
b_bavg = pm.Normal('b_bavg', 0, 2)

In [ ]:
# linear predictor
pitch_idx = df['pitch_idx'].values
pressure = df['Pressure'].values
bavg = (df['Batter_Avg'] - df['Batter_Avg'].mean()) / df['Batter_Avg'].std()

logit_p = a[pitch_idx] + bp[pitch_idx] * pressure + b_bavg * bavg
p = pm.math.sigmoid(logit_p)
y = pm.Bernoulli('y', p=p, observed=df['Is_Wicket'].values)

idata_pitch = pm.sample(draws=2000, tune=1000, target_accept=0.9)

#  Forest plot: bp (pressure slope) by pitch with 94% HDI

In [ ]:
az.plot_forest(idata_pitch, var_names=['bp'], combined=True, hdi_prob=0.94)
plt.title('Posterior: Pressure Effect (bp) by Pitch Type (94% HDI)')
plt.xlabel('log-odds change per pressure unit')
plt.tight_layout()
plt.show()

# Bar plot: bp means with HDI error bars across pitches

In [ ]:
bp_post = idata_pitch.posterior['bp'].values  # shape: (chains, draws, pitches)
bp_mean = bp_post.mean(axis=(0,1))
bp_hdi = az.hdi(idata_pitch.posterior['bp'], hdi_prob=0.94).values  # took pitches, 2)

plt.figure(figsize=(10,6))
x_pos = np.arange(len(pitch_names))
plt.bar(x_pos, bp_mean, yerr=[bp_mean-bp_hdi[:,0], bp_hdi[:,1]-bp_mean], 
        capsize=5, alpha=0.7, color='teal', edgecolor='black')
plt.xticks(x_pos, pitch_names, rotation=45)
plt.ylabel('Pressure slope (bp) posterior mean')
plt.title('Pressure Effect by Pitch Type (94% HDI)')
plt.axhline(0, color='red', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Posterior predictive check per pitch

In [ ]:
fig, axes = plt.subplots(2, len(pitch_names)//2 + 1, figsize=(15,8))
axes = axes.flatten()
for i, pitch in enumerate(pitch_names[:len(axes)]):
    pitch_mask = df['pitch_idx'] == i
    obs_wickets = df.loc[pitch_mask, 'Is_Wicket'].value_counts(normalize=True)

### Simulate posterior predictive

In [ ]:

a_pitch = idata_pitch.posterior['a'][:,:,:,i].flatten()
bp_pitch = idata_pitch.posterior['bp'][:,:,:,i].flatten()
b_bavg_post = idata_pitch.posterior['b_bavg'].flatten()
pressure_vals = df.loc[pitch_mask, 'Pressure'].values
bavg_vals = (df.loc[pitch_mask, 'Batter_Avg'] - df['Batter_Avg'].mean()) / df['Batter_Avg'].std()
    
logit_sim = np.random.normal(a_pitch[:,None], 0.1, size=(len(a_pitch), len(pressure_vals))) + \
                bp_pitch[:,None] * pressure_vals[None,:] + b_bavg_post[:,None] * bavg_vals[None,:]
p_sim = 1 / (1 + np.exp(-logit_sim))
y_sim = np.random.binomial(1, p_sim)
pred_prop = y_sim.mean(axis=0).mean()  # Aggregate
    
axes[i].bar([0,1], [obs_wickets.get(1,0), pred_prop], 
                color=['orange', 'blue'], alpha=0.8, label=['Observed', 'Predicted'])
axes[i].set_title(f'{pitch}')
axes[i].set_ylim(0, 0.2)
axes[-1].axis('off')
plt.suptitle('Posterior Predictive Check: Observed vs Predicted Wicket Rates')
plt.tight_layout()
plt.show()

# Heatmap: pitch × dot_bin observed vs predicted

In [ ]:
cross_data = df.groupby(['pitch_idx', 'dot_bin'])['Is_Wicket'].agg(['mean', 'count']).reset_index()
cross_data['pitch_name'] = [pitch_names[i] for i in cross_data['pitch_idx']]

plt.figure(figsize=(12,8))
pivot_obs = cross_data.pivot(index='pitch_name', columns='dot_bin', values='mean')
sns.heatmap(pivot_obs, annot=True, fmt='.3f', cmap='Reds', cbar_kws={'label': 'P(Wicket)'})
plt.title('Observed Wicket Probability: Pitch × Dot Sequence Length')
plt.xlabel('Previous Dot Balls')
plt.ylabel('Pitch Type')
plt.tight_layout()
plt.show()

## Key insights: 
- bp forest shows pitch-specific pressure sensitivity; PPC validates model fit per pitch; heatmap reveals dot-pressure interactions by surface for IPL bowler strategy.

## Interpretation:
- If sigma_bp is small, → pressure effect is consistent across pitch types.

- If sigma_bp is large and some bp[pitch] HDIs exclude zero while others don't → pitch moderates pressure effect and the coach's intuition 'mental' effect interacts with surface.

### Sequence plots: Sample 10 overs as state sequences

In [ ]:
overs_sample = df.groupby(['Match_ID', 'Over']).apply(lambda g: ''.join(g['State'])).sample(10)
fig, axes = plt.subplots(5, 2, figsize=(15, 10))
axes = axes.flatten()
for i, (idx, seq) in enumerate(overs_sample.items()):
    states = list(seq)
    axes[i].imshow([[1 if s=='D' else 2 if s=='R' else 3 for s in states]], 
                   cmap='Set1', aspect='auto')
    axes[i].set_xticks(range(len(states)))
    axes[i].set_xticklabels(states, rotation=45)
    axes[i].set_yticks([])
    axes[i].set_title(f'Over {idx[1]} (Match {idx[0]})')
plt.suptitle('Sample Over Sequences (D=Dot, R=Run, W=Wicket)')
plt.tight_layout()
plt.show()

### Survival plot: Kaplan-Meier for dot sequences until wicket

In [ ]:
kmf = KaplanMeierFitter()
dot_data = df[df['prev_dot_run_len'] > 0].copy()
kmf.fit(dot_data['prev_dot_run_len'], event_observed=dot_data['Is_Wicket'])
plt.figure(figsize=(10,6))
kmf.plot_survival_function()
plt.title('Survival Probability: No Wicket After t Consecutive Dots')
plt.xlabel('Consecutive Dot Balls')
plt.ylabel('Survival Probability (No Wicket)')
plt.grid(True, alpha=0.3)
plt.show()

# Conditional heatmap: P(Wicket | prev_dot_run_len, Pitch)

In [ ]:
pivot_cond = df.groupby(['Pitch_Simple', 'prev_dot_run_len'])['Is_Wicket'].mean().unstack(fill_value=0)
plt.figure(figsize=(12,6))
sns.heatmap(pivot_cond, annot=True, fmt='.3f', cmap='Reds', 
            cbar_kws={'label': 'P(Wicket)'})
plt.title('P(Wicket | Pitch × Previous Dot Sequence Length)')
plt.xlabel('Previous Dots')
plt.ylabel('Pitch Type')
plt.tight_layout()
plt.show()


# Interaction plots: P(wicket) vs prev_dot_run_len by bowler

In [ ]:
top_bowlers = df['Bowler'].value_counts().head(2).index
fig, ax = plt.subplots(figsize=(10,6))
for bowler in top_bowlers:
    bowler_data = df[df['Bowler'] == bowler]
    prob_bowler = bowler_data.groupby('prev_dot_run_len')['Is_Wicket'].mean()
    ax.plot(prob_bowler.index, prob_bowler.values, 'o-', linewidth=2, 
            label=f"{bowler} (n={len(bowler_data):,})")
ax.set_xlabel('Previous Consecutive Dot Balls')
ax.set_ylabel('P(Wicket)')
ax.set_title('Wicket Probability vs Dot Pressure by Top Bowlers')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Visual insights: 
- Sequences reveal dot streaks → wicket patterns; survival shows dot pressure decay; heatmap exposes pitch-specific pressure effects; bowler comparison identifies dot-building specialists.

#### Print short interpretative sentence you can paste into report

In [ ]:
 
interpretation = f"""
Interpretation:
- Pressure coefficient (log-odds) for Bowler A: mean {effect_A.mean():.4f}, 94% HDI [{hdi_A[0]:.4f}, {hdi_A[1]:.4f}]
- Pressure coefficient (log-odds) for Bowler B: mean {effect_B.mean():.4f}, 94% HDI [{hdi_B[0]:.4f}, {hdi_B[1]:.4f}]
Concluding sentence: The model estimates the effect of 'pressure' (prev-ball dot) on the log-odds of taking a wicket.
If the HDIs are separated as above, it provides evidence for a differential 'killer instinct' between the bowlers.
"""
print(interpretation)

### Decision rules

In [ ]:
# Interpret coefficients in log-odds:
#   positive -> pressure increases wicket probability (on next ball)
#   negative -> pressure decreases wicket probability
A_pos = (sum_A['hdi_low'] > 0)
B_pos = (sum_B['hdi_low'] > 0)

In [ ]:
# Check non-overlap: if lower bound of B > upper bound of A, B is clearly greater
B_clearly_gt_A = (sum_B['hdi_low'] > sum_A['hdi_high'])
A_clearly_gt_B = (sum_A['hdi_low'] > sum_B['hdi_high'])

In [ ]:
# Compose human-readable conclusion
if B_clearly_gt_A:
    decision_text = "Recommend BUY Bowler B — Bowler B's pressure effect (log-odds) is clearly larger than Bowler A's (no HDI overlap)."
elif A_clearly_gt_B:
    decision_text = "Recommend BUY Bowler A — Bowler A's pressure effect (log-odds) is clearly larger than Bowler B's (no HDI overlap)."
else:
    # If one HDI excludes zero and the other includes zero
    if B_pos and not A_pos:
        decision_text = ("Recommend BUY Bowler B — Bowler B shows a positive pressure->wicket effect "
                         "while Bowler A's effect is uncertain (HDI crosses zero).")
    elif A_pos and not B_pos:
        decision_text = ("Recommend BUY Bowler A — Bowler A shows a positive pressure->wicket effect "
                         "while Bowler B's effect is uncertain (HDI crosses zero).")
    elif A_pos and B_pos:
        decision_text = ("Both bowlers show positive pressure effects (94% HDIs above zero) — "
                         "no decisive winner because HDIs overlap; consider secondary metrics (economy, consistency).")
    else:
        decision_text = ("No strong evidence that pressure increases wicket probability for either bowler "
                         "(both 94% HDIs include zero). Recommendation: inconclusive — collect more data or run hierarchical model.")

# ---------- Print results ----------
print("=== Pressure Effect Summary (log-odds) ===")
print(f"Bowler A: mean={sum_A['mean']:.4f}, sd={sum_A['sd']:.4f}, 94% HDI=[{sum_A['hdi_low']:.4f}, {sum_A['hdi_high']:.4f}]")
print(f"Bowler B: mean={sum_B['mean']:.4f}, sd={sum_B['sd']:.4f}, 94% HDI=[{sum_B['hdi_low']:.4f}, {sum_B['hdi_high']:.4f}]")
print("\nDecision:")
print(decision_text)

In [ ]:
# Print short interpretative sentence you can paste into report
interpretation = f"""
Interpretation:
- Pressure coefficient (log-odds) for Bowler A: mean {effect_A.mean():.4f}, 94% HDI [{hdi_A[0]:.4f}, {hdi_A[1]:.4f}]
- Pressure coefficient (log-odds) for Bowler B: mean {effect_B.mean():.4f}, 94% HDI [{hdi_B[0]:.4f}, {hdi_B[1]:.4f}]
Concluding sentence: The model estimates the effect of 'pressure' (prev-ball dot) on the log-odds of taking a wicket.
If the HDIs are separated as above, it provides evidence for a differential 'killer instinct' between the bowlers.
"""
print(interpretation)

### Stored in pickle file

In [ ]:
print("Saved: ipl_death_overs.pkl + killer_instinct_results.pkl")

In [ ]:
# After your IPL preprocessing (from Analytics-1.ipynb)
df_death.to_pickle('ipl_death_overs_processed.pkl')

 